In [1]:
import tskit
import numpy as np
import gaiapy as gp
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from tqdm import tqdm
import time
import multiprocessing as mp
import os

In [3]:
#define the functions used below 

def locations(ts):
    nodes = ts.nodes()
    locs_array = []
    for node in nodes:
        if node.individual != -1:
            ind = ts.individual(node.individual)
            x = ind.location[0]
            y = ind.location[1]
            is_sample = node.is_sample()
            locs_array.append(node.id)
            locs_array.append(is_sample)
            locs_array.append(x)
            locs_array.append(y)
        
    locs_array = np.array(locs_array)
    locs = locs_array.reshape(-1, 4)
    return locs

def findUnary(ts):
    unary_nodes = np.zeros(ts.num_nodes) # binary vector specifying if a node is unary or not anywhere on the tree sequence
    for tree in ts.trees():
        num_children = tree.num_children_array[:ts.num_nodes]
        is_unary = num_children == 1
        for i, condition in enumerate(is_unary):
            if is_unary[i] == True:
                unary_nodes[i] = 1
    mask = unary_nodes == 1
    #mask, unary_nodes[mask]
    unary_list = np.where(mask)
    unary_indices = unary_list[0]
    return unary_indices

def find_children(unary_indices, node_stats_df):
    children = set()
    for node_id in unary_indices:
        n = node_stats_df.loc[node_id, 'distinct_children']
        if n is None or len(n) == 0:
            continue
        n = n[n != -1]
        children.update(n)
    return np.array(list(children))

def find_parents(unary_indices, node_stats_df):
    parents = set()
    for node_id in unary_indices:
        n = node_stats_df.loc[node_id, 'distinct_parents']
        if n is None or len(n) == 0:
            continue
        n = n[n != -1]
        parents.update(n)
    return np.array(list(parents))

def node_spans(ts, include_missing=False):
    """
    Returns the array of "node spans", i.e., the `j`th entry gives
    the total span over which node `j` is in the tree sequence.
    Sample nodes that are isolated are "missing data"; inclusion
    of these spans are controlled by `include_missing`. (If
    `include_missing` is `True` then the span of each sample is
    always equal to the sequence length.)

    :param bool include_missing: Whether to include spans of nodes
        on which they have missing data.
    """
    child_spans = np.bincount(
        ts.edges_child,
        weights=ts.edges_right - ts.edges_left,
        minlength=ts.num_nodes,
    )
    for t in ts.trees():
        span = t.span
        for r in t.roots:
            # do this check to exempt 'missing data'
            if include_missing or (t.num_children(r) > 0):
                child_spans[r] += span
    return child_spans




def get_span_stats(ts, ets):
    num_ets_nodes = ets.num_nodes
    added_span = np.zeros(num_ets_nodes)
    wrong_added_span = np.zeros(num_ets_nodes)
    max_slim = max(n.metadata["slim_id"] for n in ts.nodes())
    node_map = np.full(max_slim + 1, -1, dtype=np.int32)
    for n in ts.nodes():
        node_map[n.metadata["slim_id"]] = n.id
    ets_slim_ids = np.array([n.metadata["slim_id"] for n in ets.nodes()])
    
    for interval, t, et in ts.coiterate(ets):
        span = interval[1] - interval[0]
        current_t_nodes = set(t.nodes()) 

        for n in et.nodes():
            n_children = et.num_children(n)
            
            if n_children == 1:
                added_span[n] += span
    
            on = node_map[ets_slim_ids[n]]
            
            if on not in current_t_nodes:
                if n_children != 1:
                    raise AssertionError(f"Error at {interval}: node {n} has {n_children} children")
                
                wrong_added_span[n] += span
    print("done span stats")

    return added_span, wrong_added_span

# def get_span_stats(ts, ets):
#     node_map = {}
#     added_span = np.zeros(ets.num_nodes)
#     wrong_added_span = np.zeros(ets.num_nodes)
#     for n in ts.nodes():

#         slim_id = n.metadata["slim_id"]
#         assert slim_id not in node_map
#         node_map[slim_id] = n.id

#     for interval, t, et in ts.coiterate(ets):
#         interval_length = interval[1] - interval[0]
#         t_nodes = list(t.nodes())
#         #et_nodes = list(et.nodes())
#         for n in et.nodes():
#             # print("et nodes", et_nodes)
#             if et.num_children(n) == 1:
#                 added_span[n] += interval_length
#             #on = node_map[n]
#             # for x in on:
#             node = ets.node(n)
#             on = node_map[node.metadata["slim_id"]]
#             if on not in t_nodes:
#                 assert et.num_children(n) == 1, print(interval, n, et.num_children(n), et.time(n))
#                 wrong_added_span[n] += interval_length
#                 #print("added ", interval_length, " to wrong_added_span")
#     # assert not np.array_equal(added_span, wrong_added_span)
#     print("done span stats")
#     return added_span, wrong_added_span

def get_node_stats(ts):
    # node_ids = np.array([n.id for n in ts.nodes()])
    node_ids = np.arange(ts.num_nodes)
    # is_sample = np.isin(node_ids, ts.samples()).astype(int)
    
    children_dict = {}
    parents_dict = {}
    is_root = {}
    for nid in node_ids:
        children_dict[nid] = set()
        parents_dict[nid] = set()
        is_root[nid] = 0   

        #parallelize this for loop below, can go through trees independelty - run this seperatly?     

    for tree in tqdm(ts.trees()): 
        for nid in tree.nodes():
            if tree.is_root(nid):
                is_root[nid] = 1
            children_dict[nid].update(tree.children(nid))
            parents_dict[nid].add(tree.parent(nid))

    # distinct_children = list(map(children_dict.values(), list()))
    # distinct_parents = list(map(parents_dict.values(), list()))
    distinct_children = []
    distinct_parents = []
    
    for nid in node_ids:

        as_child_list = list(children_dict[nid])
        as_child_array = np.array(as_child_list)
        distinct_children.append(as_child_array)

        as_parent_list = list(parents_dict[nid])
        as_parent_array = np.array(as_parent_list)
        distinct_parents.append(as_parent_array)


    data_dict = pd.DataFrame({
        'id': node_ids,
        'num_children': [len(children_dict[nid]) for nid in node_ids],
        'distinct_children': distinct_children,
        'distinct_parents': distinct_parents,
        'num_parents': [len(parents_dict[nid]) for nid in node_ids],
        # 'is_sample': is_sample,
        'is_root': np.array([is_root[nid] for nid in node_ids])
    })

    return data_dict


In [4]:
# define the worker functions that independently get all the diff things i neeed that take forever 
def worker_get_node_stats(ts_path, result_queue):
   
    ts = tskit.load(ts_path)
    result_queue.put(('node_stats', get_node_stats(ts)))
    print("done worker get node stats")
    return

def worker_get_span_stats(ts_path, ets_path, result_queue):
   
    ts = tskit.load(ts_path)
    ets = tskit.load(ets_path)
    result_queue.put(('span_stats', get_span_stats(ts, ets)))
    print("done worker span stats")
    return

def worker_get_node_spans(sts_path, ets_path, result_queue):

    sts = tskit.load(sts_path)
    ets = tskit.load(ets_path)
    result_queue.put(('node_spans', (node_spans(sts), node_spans(ets))))
    print("done with worker node spans")
    return

def worker_gaia(sts_path, ets_path, sample_locations, result_queue):
    sts = tskit.load(sts_path)
    ets = tskit.load(ets_path)

    # sts_q_start = time.time()
    sts_mpr = gp.quadratic_mpr(sts, sample_locations)
    # sts_q_end = time.time()
    # sts_q_run = sts_q_end - sts_q_start

    # sts_min_start = time.time()
    sts_map_x = gp.quadratic_mpr_minimize(sts_mpr)
    # sts_min_end = time.time()
    # sts_min_run = sts_min_end - sts_min_start

    # ets_q_start = time.time()
    ets_mpr = gp.quadratic_mpr(ets, sample_locations)
    # ets_q_end = time.time()
    # ets_q_run = ets_q_end - ets_q_start

    # ets_min_start = time.time()
    ets_map_x = gp.quadratic_mpr_minimize(ets_mpr)
    # ets_min_end = time.time()
    # ets_min_run = ets_min_end - ets_min_start
    

    result_queue.put(('gaia', (sts_map_x, ets_map_x)))
    print("domne with worker gaia ")
    return



In [5]:
def getAccOut(ts_path, out_prefix, sigma, rep):
    ts = tskit.load(ts_path)
    sts = ts.simplify()
    ets = sts.extend_haplotypes()

    sts_path = f"{out_prefix}_sts_temp.trees"
    ets_path  = f"{out_prefix}_ets_temp.trees"
    sts.dump(sts_path)
    ets.dump(ets_path)

    sts_num_trees = sts.num_trees
    ets_num_trees  = ets.num_trees
    sts_num_edges = sts.num_edges
    ets_num_edges  = ets.num_edges

    locs = locations(ets)
    sample_locations = locs[locs[:, 1] == 1][:, [0, 2, 3]]
    sample_centroid = np.mean(sample_locations[:, 1:], axis=0)

    print("done with getting stuff before attempting multiprocessing")

    result_queue = mp.Queue()

    processes = [
        mp.Process(target=worker_get_node_stats, args=(ts_path, result_queue)),
        mp.Process(target=worker_get_span_stats, args=(ts_path, ets_path, result_queue)),
        mp.Process(target=worker_get_node_spans, args=(sts_path, ets_path, result_queue)),
        mp.Process(target=worker_gaia, args=(sts_path, ets_path, sample_locations, result_queue)),
    ]

    # for process_1 in processes: 
    #     if process_1.is_alive():
    #         process_1.terminate()
    # return

    # # POOL 4, 

    for p in processes:
        p.start()

    results = {}
    for i in processes:
        key, val = result_queue.get()
        results[key] = val

    for p in processes:
        p.join()

    print("done multi processing")

    node_stats_df = results['node_stats']
    total_added_span, wrongly_added_span = results['span_stats']
    sts_spans, ets_spans = results['node_spans']
    sts_map_x, ets_map_x = results['gaia']

    os.remove(sts_path)
    os.remove(ets_path)

    print("done like getting results from multiprocessing")

    unary_indices = findUnary(ets)
    children = find_children(unary_indices, node_stats_df)
    parents = find_parents(unary_indices,  node_stats_df)


    is_unary = np.isin(locs[:, 0], unary_indices)
    is_ancestor = np.isin(locs[:, 0], unary_indices)
    is_child = np.isin(locs[:,0], children)
    is_parent = np.isin(locs[:,0], parents)
    unary_nodes = is_unary & is_ancestor 
    child_nodes = is_child
    parent_nodes = is_parent

    unary_locations = locs[unary_nodes][:, [0, 2, 3]]
    child_locations = locs[child_nodes][:, [0, 2, 3]]
    parent_locations = locs[parent_nodes][:, [0, 2, 3]]

    unary_sts_e = np.sqrt(np.sum((sts_map_x[unary_nodes] - unary_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))
    unary_ets_e = np.sqrt(np.sum((ets_map_x[unary_nodes] - unary_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))

    child_sts_e = np.sqrt(np.sum((sts_map_x[child_nodes] - child_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))
    child_ets_e = np.sqrt(np.sum((ets_map_x[child_nodes] - child_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))

    parent_sts_e = np.sqrt(np.sum((sts_map_x[parent_nodes] - parent_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))
    parent_ets_e = np.sqrt(np.sum((ets_map_x[parent_nodes] - parent_locations[:, 1:2])**2, axis=1)) / np.max(pdist(sample_locations[:, 1:2]))
  
    unary_dist_from_sample_centroid0 = np.sqrt(np.sum((unary_locations[:, 1:3] - sample_centroid)**2, axis=1))
    child_dist_from_sample_centroid0 = np.sqrt(np.sum((child_locations[:, 1:3] - sample_centroid)**2, axis=1))
    parent_dist_from_sample_centroid0 = np.sqrt(np.sum((parent_locations[:, 1:3] - sample_centroid)**2, axis=1))

    unary_sts_dist_from_sample_centroid = np.sqrt(np.sum((sts_map_x[unary_nodes] - sample_centroid)**2, axis=1))
    unary_ets_dist_from_sample_centroid = np.sqrt(np.sum((ets_map_x[unary_nodes] - sample_centroid)**2, axis=1))

    child_sts_dist_from_sample_centroid = np.sqrt(np.sum((sts_map_x[child_nodes] - sample_centroid)**2, axis=1))
    child_ets_dist_from_sample_centroid = np.sqrt(np.sum((ets_map_x[child_nodes] - sample_centroid)**2, axis=1))

    parent_sts_dist_from_sample_centroid = np.sqrt(np.sum((sts_map_x[parent_nodes] - sample_centroid)**2, axis=1))
    parent_ets_dist_from_sample_centroid = np.sqrt(np.sum((ets_map_x[parent_nodes] - sample_centroid)**2, axis=1))

    unary_node_ids = locs[unary_nodes, 0]
    unary_node_times = ets.nodes_time[unary_nodes]

    child_node_ids = locs[child_nodes, 0]
    child_node_times = ets.nodes_time[child_nodes]

    parent_node_ids = locs[parent_nodes, 0]
    parent_node_times = ets.nodes_time[parent_nodes]

    sts_spans = sts_spans[unary_nodes]
    ets_spans = ets_spans[unary_nodes]

    total_added_span = total_added_span[unary_nodes]
    wrong_added_span = wrongly_added_span[unary_nodes]

    unary_df = pd.DataFrame({
        'unary_node_id': unary_node_ids,
        'unary_node_time': unary_node_times,
        'unary_sts_error': unary_sts_e,
        'unary_ets_error': unary_ets_e,
        'unary_dist_from_sample_centroid0': unary_dist_from_sample_centroid0,
        'unary_sts_dist_from_sample_centroid': unary_sts_dist_from_sample_centroid,
        'unary_ets_dist_from_sample_centroid': unary_ets_dist_from_sample_centroid,
        'added_span': total_added_span,
        'wrongly_added_span': wrong_added_span,
        'sts_span': sts_spans,
        'ets_span': ets_spans
    })

    child_df = pd.DataFrame({
        'child_node_id': child_node_ids,
        'child_node_time': child_node_times,
        'child_sts_error': child_sts_e,
        'child_ets_error': child_ets_e,
        'child_dist_from_sample_centroid0': child_dist_from_sample_centroid0,
        'child_sts_dist_from_sample_centroid': child_sts_dist_from_sample_centroid,
        'child_ets_dist_from_sample_centroid': child_ets_dist_from_sample_centroid
    })

    parent_df = pd.DataFrame({
        'parent_node_id': parent_node_ids,
        'parent_node_time': parent_node_times,
        'parent_sts_error': parent_sts_e,
        'parent_ets_error': parent_ets_e,
        'parent_dist_from_sample_centroid0': parent_dist_from_sample_centroid0,
        'parent_sts_dist_from_sample_centroid': parent_sts_dist_from_sample_centroid,
        'parent_ets_dist_from_sample_centroid': parent_ets_dist_from_sample_centroid

    })

    static_df = pd.DataFrame({
        'sigma': [sigma],
        'rep': [rep],
        'sts_num_trees': [sts_num_trees],
        'ets_num_trees': [ets_num_trees], 
        'sts_num_edges': [sts_num_edges],
        'ets_num_edges': [ets_num_edges]
    })

    # time_df = pd.DataFrame({
    #     'simp_quad_mpr': [sts_q_run], 
    #     'simp_minimize': [sts_min_run],
    #     'ets_quad_mpr': [ets_q_run],
    #     'ets_minimize': [ets_min_run]
    # })


    unary_df.to_csv(f"{out_prefix}_unary_results.csv",
                      mode='a', header=True, index=False)
    
    child_df.to_csv(f"{out_prefix}_child_results.csv",
                      mode='a', header=True, index=False)
    
    parent_df.to_csv(f"{out_prefix}_parent_results.csv",
                      mode='a', header=True, index=False)
    
    static_df.to_csv(f"{out_prefix}_static_info.csv",
                      mode='a', header=True, index=False)
    
    node_stats_df.to_csv(f"{out_prefix}_node_stats.csv",
                      mode='a', header=True, index=False)

    # time_df.to_csv(f"{out_prefix}_time.csv",
    #                mode='a', header=True, index=False)
    

    return unary_df, child_df, parent_df, static_df, node_stats_df


In [7]:




ts_path = 'tree-files/tree-S1.7-R11.trees'
out_prefix = '1.7-R11_'
sigma = 1.7
rep = 11
getAccOut(ts_path, out_prefix, sigma, rep)

ts_path = 'tree-files/tree-S1.7-R12.trees'
out_prefix = '1.7-R12_'
sigma = 1.7
rep = 12
getAccOut(ts_path, out_prefix, sigma, rep)

ts_path = 'tree-files/tree-S1.7-R13.trees'
out_prefix = '1.7-R13_'
sigma = 1.7
rep = 13
getAccOut(ts_path, out_prefix, sigma, rep)

ts_path = 'tree-files/tree-S1.7-R14.trees'
out_prefix = '1.7-R14_'
sigma = 1.7
rep = 14
getAccOut(ts_path, out_prefix, sigma, rep)

ts_path = 'tree-files/tree-S1.7-R15.trees'
out_prefix = '1.7-R15_'
sigma = 1.7
rep = 15
getAccOut(ts_path, out_prefix, sigma, rep)

ts_path = 'tree-files/tree-S1.7-R16.trees'
out_prefix = '1.7-R16_'
sigma = 1.7
rep = 16
getAccOut(ts_path, out_prefix, sigma, rep)












# ts_path = 'tree-files/tree-S1.1-R19.trees'
# out_prefix = '1.1-R19_'
# sigma = 1.1
# rep = 19
# getAccOut(ts_path, out_prefix, sigma, rep)


# ts_path = 'tree-files/tree-S1.1-R20.trees'
# out_prefix = '1.1-R20_'
# sigma = 1.1
# rep = 20
# getAccOut(ts_path, out_prefix, sigma, rep)

# ts_path = 'tree-files/tree-S1.1-R17.trees'
# out_prefix = '1.1-R17_'
# sigma = 1.1
# rep = 17
# getAccOut(ts_path, out_prefix, sigma, rep)





# ts_path = 'tree-files/tree-S0.5-R17.trees'
# out_prefix = '0.5-R17_'
# sigma = 0.5
# rep = 17
# getAccOut(ts_path, out_prefix, sigma, rep)








done with getting stuff before attempting multiprocessing
done with worker node spans


 17%|█▋        | 7132/43154 [16:36<1:32:15,  6.51it/s]

done span stats
done worker span stats


100%|██████████| 43154/43154 [1:40:53<00:00,  7.13it/s]


done worker get node stats
domne with worker gaia 
done multi processing
done like getting results from multiprocessing
done with getting stuff before attempting multiprocessing
done with worker node spans


...

 17%|█▋        | 6922/39781 [16:25<1:12:29,  7.55it/s]

done span stats
done worker span stats


100%|██████████| 39781/39781 [1:24:35<00:00,  7.84it/s]


done worker get node stats
domne with worker gaia 
done multi processing
done like getting results from multiprocessing


(       unary_node_id  unary_node_time  unary_sts_error  unary_ets_error  \
 0             3036.0              1.0         3.151960         3.146723   
 1             3037.0              1.0         3.151678         3.153668   
 2             3038.0              1.0         4.606790         4.607906   
 3             3039.0              1.0         4.582220         4.578853   
 4             3040.0              1.0         3.679559         3.677991   
 ...              ...              ...              ...              ...   
 10788        14289.0           2988.0         3.394216         3.322301   
 10789        14290.0           2990.0         3.264560         3.347052   
 10790        14300.0           3033.0         4.037491         4.043256   
 10791        14302.0           3036.0         3.941224         3.863995   
 10792        14307.0           3096.0         3.898264         3.931679   
 
        unary_dist_from_sample_centroid0  unary_sts_dist_from_sample_centroid  \
 0   

In [ ]:
ts_path = 'tree-files/tree-S1.1-R1.trees'
out_prefix = 'S1.1-R0_'
sigma = 1.1
rep = 1
getAccOut(ts_path, out_prefix, sigma, rep)



ts_path = 'tree-files/tree-S1.1-R2.trees'
out_prefix = 'S1.1-R2_'
sigma = 1.1
rep = 2
getAccOut(ts_path, out_prefix, sigma, rep)


ts_path = 'tree-files/tree-S1.1-R3.trees'
out_prefix = 'S1.1-R3_'
sigma = 1.1
rep = 3
getAccOut(ts_path, out_prefix, sigma, rep)

done with getting stuff before attempting multiprocessing
done with worker node spans


 17%|█▋        | 7684/45421 [18:45<1:37:58,  6.42it/s]

done worker span stats


100%|██████████| 45421/45421 [1:57:27<00:00,  6.44it/s]  


done worker get node stats
domne with worker gaia 
done multi processing
done like getting results from multiprocessing
done with getting stuff before attempting multiprocessing
done with worker node spans


...

 82%|████████▏ | 40191/48748 [1:44:43<22:44,  6.27it/s]  

domne with worker gaia 


100%|██████████| 48748/48748 [2:05:18<00:00,  6.48it/s]  


done worker get node stats
done multi processing
done like getting results from multiprocessing


(      unary_node_id  unary_node_time  unary_sts_error  unary_ets_error  \
 0            2480.0              1.0         2.039871         2.039594   
 1            2481.0              1.0         0.758886         0.758655   
 2            2482.0              1.0         0.756916         0.756583   
 3            2483.0              1.0         0.854137         0.853629   
 4            2484.0              1.0         1.836769         1.834090   
 ...             ...              ...              ...              ...   
 9108        11899.0           6780.0         1.960973         1.803241   
 9109        11900.0           6823.0         1.952803         1.774060   
 9110        11901.0           6867.0         1.929408         1.885391   
 9111        11902.0           6888.0         1.940640         1.781500   
 9112        11904.0           7093.0         1.743014         1.685272   
 
       unary_dist_from_sample_centroid0  unary_sts_dist_from_sample_centroid  \
 0                

In [ ]:

# Create a new process
process = Process()

def func(num):
    print("hello world")
    print(num)

process = Process(target=func, args=("yooo",))

process.start()

process.is_alive()

In [ ]:
def do_something():
    print("I'm going to sleep")
    time.sleep(1)
    print("I'm awake") 

process_1 = Process(target=do_something)
process_2 = Process(target=do_something)

In [ ]:
%%time

# Create new child process (Cannot run a process more than once)
new_process_1 = Process(target=do_something)
new_process_2 = Process(target=do_something)

# Starts both processes
new_process_1.start()
new_process_2.start()

new_process_1.join()
new_process_2.join()

In [ ]:
if process_1.is_alive():
    process_1.terminate() # You can also use process.kill()

if process_2.is_alive():
    process_2.terminate() # Y